# Getting Started with Unstructured Transform

You have a PDF. You hand it to a language model and ask about the table on page 8. What comes back is
a shapeless run of digits, because somewhere between the page and the prompt the rows and columns
stopped existing.

So you go and build a parsing pipeline. Layout detection, OCR for the scanned pages, something to keep
tables intact, chunking, embeddings. Weeks later you have infrastructure to maintain and you still
have not written the feature you wanted.

**Unstructured Transform is that pipeline, hosted, driven by a sentence.** Point it at a file and it
runs as many of these stages as your request needs:

| Stage | What it does |
| --- | --- |
| **Parse** | Reads the raw file into clean, structured elements |
| **Enrich** | Adds AI passes: image descriptions, OCR cleanup, tables to HTML |
| **Chunk** | Splits content into retrieval-sized pieces |
| **Embed** | Turns those chunks into vectors for semantic search |
| **Extract** | Pulls named fields straight out of the document as JSON |

You never configure any of it by hand. You say what you want and it assembles the right stages.

By the end of this notebook you will have built:

- A question-answering system over a PDF with no vector database at all
- The same thing with embeddings, for when you outgrow that
- A structured extractor that turns the document into database rows

We will use the *"Attention Is All You Need"* paper throughout, because its tables and notation are
exactly what naive text extraction destroys.

One thing worth knowing before you start: you do not need a notebook for any of this. Transform is an
MCP server, so it plugs straight into **Claude Code, Codex, or Cursor**. You add it once, sign in, and
then just talk to it. See the [setup guide](https://docs.unstructured.io/transform/install/overview).
We are using Python here because it makes every step visible.

Let's dive in!

In [1]:
%pip install -U openai requests

Note: you may need to restart the kernel to use updated packages.


You need two keys.

For Unstructured, [sign up here](https://unstructured.io/?modal=try-for-free), log in, and copy your
API key. You get 15,000 pages free every month, and this notebook uses about 45 of them.

For OpenAI, grab a key from [the API keys page](https://platform.openai.com/api-keys).

In [2]:
import json
import math
import os
import re
import time
from getpass import getpass

import requests
from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
if "UNSTRUCTURED_API_KEY" not in os.environ:
    os.environ["UNSTRUCTURED_API_KEY"] = getpass("Unstructured API key: ")

client = OpenAI()

TRANSFORM_MCP_URL = "https://mcp.transform.unstructured.io/"
MODEL = "gpt-5"
PDF_URL = "https://arxiv.org/pdf/1706.03762"  # "Attention Is All You Need"
DOC_PATH = "attention.md"

Now we connect the model to Transform. The Responses API can call a remote MCP server directly, so we
describe the server as a tool and the model does the rest.

One piece of plumbing is needed because we are in Python. Transform processes files asynchronously, so
the model has to wait while a job runs, and a hosted tool cannot pause itself. We give it a local
`wait_seconds` function it can call to do that, and `run()` below handles the back and forth.

If you work in Claude Code, Codex or Cursor, skip past this. Your agent already handles all of it, and
you would just say "parse this PDF" and get the result.

In [3]:
def transform_mcp_tool():
    """Describe the Transform MCP server as a tool the model can call."""
    return {
        "type": "mcp",
        "server_label": "unstructured_transform",
        "server_url": TRANSFORM_MCP_URL,
        "require_approval": "never",
        "headers": {"Authorization": f"Bearer {os.environ['UNSTRUCTURED_API_KEY']}"},
    }


def wait_seconds(seconds):
    """Let the model pace itself while a Transform job runs."""
    time.sleep(seconds)
    return f"Waited {seconds} seconds."


WAIT_TOOL = {
    "type": "function",
    "name": "wait_seconds",
    "description": "Wait the given number of seconds before checking the job status again.",
    "parameters": {
        "type": "object",
        "properties": {"seconds": {"type": "integer"}},
        "required": ["seconds"],
        "additionalProperties": False,
    },
}


def run(instruction):
    """Ask Transform for something. Returns the model's reply and the job results."""
    tools = [transform_mcp_tool(), WAIT_TOOL]
    resp = client.responses.create(model=MODEL, tools=tools, input=instruction)
    results = {}

    for _ in range(40):
        for item in resp.output:
            if item.type == "mcp_call" and item.output:
                try:
                    results[item.name] = json.loads(item.output)
                except ValueError:
                    pass
        calls = [i for i in resp.output if i.type == "function_call"]
        if not calls:
            break
        outputs = [
            {
                "type": "function_call_output",
                "call_id": c.call_id,
                "output": wait_seconds(**json.loads(c.arguments)),
            }
            for c in calls
        ]
        resp = client.responses.create(
            model=MODEL, tools=tools, previous_response_id=resp.id, input=outputs
        )

    # Download links carry a long signed token that expires within the hour. Shorten
    # it so the printed reply stays readable.
    reply = re.sub(r"\?token=[\w.\-]+", "?token=...", resp.output_text)
    return reply, results

## 1. Answer questions about a document, with no vector database

Before you stand up a vector store, try the version that needs no infrastructure.

Parse the document once into markdown. Then give the model two tools, a search and a line reader, and
let it navigate the file the way you would navigate an unfamiliar codebase: search for the relevant
part, read around the hit, answer from what you read.

First, the parse. Notice that we just ask.

In [4]:
reply, results = run(
    f"Parse the PDF at {PDF_URL} into markdown using the Unstructured Transform tools. "
    "It is a research paper full of tables and figures, so choose the parse approach that "
    "handles those well. Wait for the job to finish, then give me the markdown."
)
print(reply)

The parse is complete. I used the hi_res partition strategy with image_description, generative_ocr, and table_to_html enrichments to better handle figures and tables, and rendered the result as Markdown.

Download the Markdown here:
https://mcp.transform.unstructured.io/output/539b9ef7-65c6-41e3-99d5-57b57f15bdd8_1706-27779320_md.md?token=...

One-liner to save it locally:
curl -fSs -o attention_is_all_you_need.md 'https://mcp.transform.unstructured.io/output/539b9ef7-65c6-41e3-99d5-57b57f15bdd8_1706-27779320_md.md?token=...'

Details:
- Size: ~44 KB
- Expires: 2026-07-28T02:51:25Z
- Re-mintable: If it expires, I can re-mint a fresh link instantly.
- Output ref (for chaining to extraction or re-rendering): u10d://output/539b9ef7-65c6-41e3-99d5-57b57f15bdd8_1706-27779320.json

Want me to also provide an HTML or TXT render, or extract sections (abstract, equations, tables) into structured JSON?


Transform returns the output as a download link rather than dumping a whole document into the model's
context. Let's fetch it and look at what happened to the results table.

In [5]:
files = results["get_job_results"]["files"]
download = requests.get(files[0]["download_url"], timeout=120)
download.raise_for_status()  # links expire after an hour, so fail loudly if this one has
markdown = download.text
open(DOC_PATH, "w", encoding="utf-8").write(markdown)

parsed_doc = files[0]["output_ref"]  # reusable handle, we come back to this twice
print(f"{len(markdown):,} characters\n")

start = markdown.find("<table")
print(markdown[start : start + 700])

44,308 characters

<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Attention</td><td>O(n² · d)</td><td>O(1)</td><td>O(1)</td></tr><tr><td>Recurrent</td><td>O(n · d²)</td><td>O(n)</td><td>O(n)</td></tr><tr><td>Convolutional</td><td>O(k · n · d²)</td><td>O(1)</td><td>O(log_k(n))</td></tr><tr><td>Self-Attention (restricted)</td><td>O(r · n · d)</td><td>O(1)</td><td>O(n/r)</td></tr></tbody></table>

# 3.5 Positional Encoding

Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolu


The table came back as HTML, with its rows and columns intact. That is the difference between a table
you can query and a pile of loose numbers, and it is what makes the second question below possible.

Now the two tools. This is the entire retrieval layer.

In [6]:
def search_document(query):
    """Find lines in the document containing a phrase."""
    lines = open(DOC_PATH, encoding="utf-8").read().splitlines()
    hits = [f"L{i + 1}: {line}" for i, line in enumerate(lines) if query.lower() in line.lower()]
    return "\n".join(hits[:25]) if hits else "No matches."


def read_lines(start, end):
    """Read an inclusive range of lines from the document."""
    lines = open(DOC_PATH, encoding="utf-8").read().splitlines()
    start, end = max(1, int(start)), min(len(lines), int(end))
    return "\n".join(f"L{i}: {lines[i - 1]}" for i in range(start, end + 1))


TOOL_FUNCS = {"search_document": search_document, "read_lines": read_lines}

TOOLS = [
    {
        "type": "function",
        "name": "search_document",
        "description": "Search the document for a phrase. Returns matching lines with line numbers.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "read_lines",
        "description": "Read an inclusive range of lines from the document.",
        "parameters": {
            "type": "object",
            "properties": {"start": {"type": "integer"}, "end": {"type": "integer"}},
            "required": ["start", "end"],
            "additionalProperties": False,
        },
    },
]

In [7]:
SYSTEM = (
    "You answer questions about a document you can only reach through the search_document and "
    "read_lines tools. Search to find the relevant part, read it, then answer from it alone. "
    "End with a line: 'Source: <the section or table you used>'."
)


def ask(question):
    """Answer a question by searching and reading the parsed document."""
    resp = client.responses.create(
        model=MODEL,
        tools=TOOLS,
        input=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question},
        ],
    )
    for _ in range(8):
        calls = [i for i in resp.output if i.type == "function_call"]
        if not calls:
            break
        outputs = [
            {
                "type": "function_call_output",
                "call_id": c.call_id,
                "output": TOOL_FUNCS[c.name](**json.loads(c.arguments)),
            }
            for c in calls
        ]
        resp = client.responses.create(
            model=MODEL, tools=TOOLS, previous_response_id=resp.id, input=outputs
        )
    return resp.output_text

In [8]:
print(ask("What positional encoding does the model use, and why did the authors choose it?"))

It uses fixed sinusoidal positional encodings (sine and cosine at different frequencies) added to the input embeddings. The authors chose this because it lets the model easily attend by relative positions (PEpos+k is a linear function of PEpos) and may allow extrapolation to sequence lengths longer than those seen in training; they observed similar performance to learned positional embeddings but preferred the sinusoidal version for this extrapolation property.

Source: 3.5 Positional Encoding


In [9]:
print(ask("What BLEU score did the ConvS2S Ensemble model reach on English-to-French?"))

41.29 BLEU

Source: Table 2


That second number, 41.29, appears nowhere in the paper's prose. It exists only inside a cell of the
results table, so being able to answer the question at all is the proof that the table survived
parsing.

No vector database, no chunking decisions, no index to keep in sync, and every answer cites where it
came from. For a handful of documents this is often all you need.

It stops scaling when search stops finding things. Grep either hits or it does not, so a user who
phrases a question nothing like the text gets nothing, and there is no way to rank results across a
thousand documents. That is what embeddings fix.

## 2. Add embeddings for semantic search

Embedding turns each chunk into a vector, a numeric fingerprint of its meaning, so a search matches on
meaning instead of characters.

Transform chunks and embeds for you, with OpenAI, Azure OpenAI, or Bedrock models (Titan and Cohere).
We will use `text-embedding-3-small`.

We already parsed this paper, so we hand Transform that same handle instead of paying to read the PDF
again.

In [10]:
EMBED_MODEL = "text-embedding-3-small"

reply, results = run(
    f"Take the already-parsed document at {parsed_doc} and chunk it by title into pieces of at "
    f"most 800 characters, then embed those chunks with OpenAI {EMBED_MODEL}. "
    "Wait for the job, then give me the results as JSON."
)

In [11]:
files = results["get_job_results"]["files"]
download = requests.get(files[0]["download_url"], timeout=120)
download.raise_for_status()
elements = download.json()

chunks = [
    {
        "text": e["text"],
        "vector": e["embeddings"],
        "page": (e.get("metadata") or {}).get("page_number"),
    }
    for e in elements
    if e.get("embeddings")
]

print(f"{len(chunks)} chunks, {len(chunks[0]['vector'])} dimensions each")

70 chunks, 1536 dimensions each


Each chunk now carries a vector. The search is cosine similarity, which is a few lines of plain Python.

The one rule: embed your question with the same model you embedded the document with, or the vectors
are not comparable.

In [12]:
def cosine(a, b):
    """Similarity between two vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


def search(question, k=3):
    """Find the chunks closest in meaning to the question."""
    q = client.embeddings.create(model=EMBED_MODEL, input=question).data[0].embedding
    return sorted(chunks, key=lambda c: cosine(q, c["vector"]), reverse=True)[:k]


question = "What positional encoding does the model use, and why did the authors choose it?"

hits = search(question)
for hit in hits:
    print(f"[page {hit['page']}] {hit['text'][:100].strip()}...")

[page 6] 3.5 Positional Encoding

Since our model contains no recurrence and no convolution, in order for the...
[page 6] where pos is the position and i is the dimension. That is, each dimension of the positional encoding...
[page 9] In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This su...


In [13]:
context = "\n\n".join(f"(page {h['page']}) {h['text']}" for h in hits)

answer = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": "Answer from the passages below and cite the page numbers."},
        {"role": "user", "content": f"Passages:\n{context}\n\nQuestion: {question}"},
    ],
)
print(answer.output_text)

The model uses fixed sinusoidal positional encodings—sine and cosine functions at different frequencies—added to the input embeddings and matching the embedding dimension (d_model) (page 6). The authors chose this scheme because it lets the model easily attend by relative positions (since PEpos+k is a linear function of PEpos) and may allow extrapolation to sequence lengths longer than those seen in training (page 6). They also found learned positional embeddings performed nearly identically but retained the sinusoidal version for its extrapolation advantages (pages 6, 9).


Same answer as before, reached by ranking rather than searching for a string. The three passages it
retrieved cost a fraction of the context that reading through the file did.

Here the vectors live in a Python list, which is fine for one paper. In production you write them to a
vector store and search there. Transform's output already carries the text, vector and page metadata
per chunk, so it maps onto whatever database you use. See
[destinations](https://docs.unstructured.io/pipelines/destinations/overview).

## 3. Pull named fields out as JSON

The first two features answered questions. Sometimes you do not want an answer, you want values: the
same fields, out of every document, in a shape you can put in a database. Invoice numbers and line
items. Contract parties and dates. Or, here, the settings you would need to reproduce a paper.

Imagine you have two hundred of these papers and you want one table comparing how each model was
trained. Asking questions does not scale to that. Extracting fields does.

Extraction works on the parsed document, so we reuse that same handle a third time. And if you do not
have a schema yet, Transform will write one for you.

In [14]:
reply, results = run(
    f"Use suggest_extraction_schema_for_file on {parsed_doc} to draft a JSON extraction schema "
    "for this paper. Keep it small and flat: just the base model's architecture and training "
    "settings, one value per field, the kind of thing you would need to reproduce it. "
    "Then report the schema."
)

schema = json.loads(results["suggest_extraction_schema_for_file"]["schema"])
print(json.dumps(schema, indent=2)[:900])

{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "Transformer Base Model Configuration",
  "type": "object",
  "properties": {
    "model_name": {
      "type": "string",
      "description": "Name of the model architecture"
    },
    "architecture": {
      "type": "string",
      "description": "Type of neural network architecture"
    },
    "parameter_count": {
      "type": "integer",
      "description": "Total number of model parameters in millions"
    },
    "vocab_size": {
      "type": "integer",
      "description": "Size of the vocabulary for tokenization"
    },
    "tokenizer": {
      "type": "string",
      "description": "Tokenization method used"
    },
    "optimizer": {
      "type": "string",
      "description": "Optimization algorithm used for training"
    },
    "learning_rate_warmup_steps": {
      "type": "integer",
      "description": "N


Now extract with that schema.

In [15]:
reply, results = run(
    f"Run an extraction job on {parsed_doc} using exactly this schema: {json.dumps(schema)}. "
    "Wait for it to finish, then give me the extracted data."
)

print(json.dumps(results["get_job_results"]["files"][0], indent=2)[:1800])

{
  "filename": "1706-27779320.03762",
  "filetype": "application/pdf",
  "processed_date_utc": "2026-07-28T01:54:18.224869Z",
  "source_file_uri": "u10d://output/539b9ef7-65c6-41e3-99d5-57b57f15bdd8_1706-27779320.json",
  "extracted_data": {
    "model_name": "Transformer",
    "architecture": "Transformer",
    "parameter_count": 65,
    "vocab_size": 37000,
    "tokenizer": "byte-pair encoding",
    "optimizer": "Adam",
    "learning_rate_warmup_steps": 4000,
    "batch_size_tokens": 25000,
    "training_steps": 100000,
    "training_hardware": "8 NVIDIA P100 GPUs",
    "dropout": 0.1,
    "label_smoothing": 0.1,
    "dataset_name": "WMT 2014 English-German",
    "model_dimension": 512,
    "num_layers": 6,
    "num_attention_heads": 8,
    "feedforward_dimension": 2048,
    "attention_key_dimension": 64,
    "attention_value_dimension": 64
  }
}


The paper's training setup, as fields. Run that over a folder of papers and you have your comparison
table. From here it is an `INSERT`, not a language problem.

These values are scattered across the paper: the layer counts sit in a table, the optimizer settings in
a paragraph of section 5, the hardware in another. You did not have to know where any of them were.

Notice each result arrives wrapped with its `filename` and `source_file_uri`. When you run a batch of
invoices through this, that is what tells you which record came from which document.

One rule worth remembering: extraction can only find what the parse captured. Ask for fields that live
in a badly scanned table and you will get gaps, which is what the first two tips below are for.

## Tips

Everything above used one plain instruction per step. Here are the ones worth knowing next, in the same
form. Just ask.

**Documents that are mostly tables.** *"Parse this at high resolution, give me the tables as HTML, and
add a description of each one."* HTML keeps the structure so your code can read a table as a table. The
description makes it findable, because a grid of numbers matches a natural-language question badly. Add
both and "which quarter had the highest churn" can actually find the right table.

**Scanned, watermarked or handwritten pages.** *"This scan is messy, clean the text up with generative
OCR."* A vision model re-reads the difficult blocks. This paper has an arXiv watermark printed sideways
down the margin, which parses as `3 2 0 2 g u A 2` one character at a time; generative OCR is what
turns that back into `arXiv:1706.03762v7 [cs.CL] 2 Aug 2023`.

**Charts, diagrams and slide decks.** *"Describe the images in this deck so we can use what is in
them."* By default a chart is just an image and your agent answers as though its contents do not exist.
Image descriptions put a summary of each figure into the output, so it becomes something the model can
reason over and cite.

**Chunking that fits your documents.** *"Re-chunk that same parse by page instead, and by
similarity."* Where you split matters as much as how you parse. Titles suit documents with real
headings, pages suit forms and slides, similarity suits text with no clean structure. Point Transform at
a parse you already ran and compare, rather than re-parsing each time.

One thing this is not for: if you need a scheduled, continuous sync from a source to a destination
across thousands of documents, that is Unstructured
[Pipelines](https://docs.unstructured.io/pipelines/overview), a different product.

## Wrapping up

We parsed one PDF and used it three ways:

- Answered questions with nothing but a search and a line reader, and pulled a number out of a table
  that does not appear anywhere else in the document
- Chunked and embedded it for ranked semantic search, without re-parsing
- Extracted its training configuration into flat JSON, against a schema Transform wrote itself

Same document, same three instructions in plain English, three different shapes of output. That is the
whole idea: you describe what you need and Transform assembles the pipeline.

To go further, start with the [concepts overview](https://docs.unstructured.io/concepts/overview).
Everything here is one corner of it, and the interesting uses come from combining stages in ways no
tutorial will guess for you. Worth a look:
[partitioning strategies](https://docs.unstructured.io/concepts/partitioning),
[enrichments](https://docs.unstructured.io/concepts/enriching/overview),
[chunking](https://docs.unstructured.io/concepts/chunking),
[embedding](https://docs.unstructured.io/concepts/embedding), the
[data extractor](https://docs.unstructured.io/concepts/structured-data-extractor/data-extractor), and
the [65+ supported file types](https://docs.unstructured.io/pipelines/supported-file-types).

**Ready to point this at your own documents?** Swap the URL at the top for an invoice, a contract, a
deck, or a folder of mixed formats, and ask for what you need.
[Sign up for Unstructured](https://unstructured.io/?modal=try-for-free) if you have not already, and
come talk to us in the
[community Slack](https://join.slack.com/t/unstructuredw-kbe4326/shared_invite/zt-4399mep47-MfLsMuA6K9Dwy86BhNY1KA)
if you hit a document that still gives you trouble.